# **Adam (Adaptive Moment Estimation)**

Adam (Adaptive Moment Estimation) is widely considered the default go-to optimizer in modern deep learning. Introduced by Diederik Kingma and Jimmy Ba in 2014, Adam combines the two most powerful ideas in optimization into a single, unified algorithm:

1. **Momentum (1st Moment):** Accelerates search along consistent directions by tracking the moving average of raw past gradients ($m_t$).
2. **RMSProp (2nd Moment):** Adapts the learning rate for each parameter individually by tracking the moving average of past squared gradients ($v_t$).

On top of combining Momentum and RMSProp, Adam introduces a critical feature that both previous algorithms lacked: **Bias Correction**. This fixes the zero-initialization bias in early steps, ensuring stable and reliable parameter updates right from step 1.

---

Adam (Adaptive Moment Estimation) is the most widely used optimizer in modern deep learning (used in Transformers, CNNs, and LLMs).

It works by combining the two best ideas from previous optimizers into one algorithm, plus a key fix called Bias Correction.

**Adam combines:**
* **Momentum (1st Moment - $m_t$):** Tracks the average of past gradients to maintain directional inertia (velocity) and slide smoothly through ravines.
* **RMSProp (2nd Moment - $v_t$):** Tracks the average of past squared gradients to adapt the learning rate per weight individually (shrinking steps on steep slopes, taking larger steps on flat plateaus).
* **Bias Correction ($\hat{m}_t, \hat{v}_t$):** Fixes the initial "slow start" problem caused by starting memory at zero ($0$).



### **How Adam is Different From Other Optimizers**

#### 1. Quick Comparison Table

| Feature | SGD | SGD + Momentum | AdaGrad | RMSProp | Adam |
| :--- | :---: | :---: | :---: | :---: | :---: |
| **Uses Directional Momentum ($m_t$)** | ❌ | ✅ | ❌ | ❌ | **Key Difference: ✅** |
| **Adaptive Per-Weight Learning Rate ($v_t$)** | ❌ | ❌ | ✅ (Cumulative) | ✅ (Moving Avg) | **Key Difference: ✅** |
| **Bias Correction ($\hat{m}_t, \hat{v}_t$)** | ❌ | ❌ | ❌ | ❌ | **Key Difference: ✅** |
| **Learning Rate Freezing Risk** | None | None | ⚠️ High (Freezes) | None | **None** |



#### 2. Detailed Differences

##### Basic SGD
* **SGD:** Uses a single, fixed global learning rate for all weights and has no memory.
* **Adam:** Gives every weight its own personalized learning rate and uses past history (momentum + variance) to adapt step sizes.

##### SGD with Momentum
* **SGD + Momentum:** Only tracks the **direction/velocity** of past gradients ($m_t$). It still applies a single global learning rate to all parameters.
* **Adam:** Tracks **both** direction ($m_t$) AND adapts the learning rate per individual weight ($v_t$).

##### AdaGrad
* **AdaGrad:** Accumulates *all* past squared gradients from step 1 to infinity ($S_t = \sum g_t^2$). The learning rate rapidly decays to **zero**, freezing training prematurely.
* **Adam:** Uses an Exponential Moving Average (fading short-term memory), ensuring the learning rate **never freezes**. Plus, Adam adds Momentum and Bias Correction.

##### RMSProp
* **RMSProp:** Only tracks the 2nd moment ($v_t$ for adaptive step sizes). It **lacks momentum** ($m_t$) and **lacks bias correction**.
* **Adam:** Takes RMSProp and adds **Momentum** (directional velocity) + **Bias Correction** (fixing zero-initialization bias in early steps).

---



## 1. The Core Intuition: Momentum + Adaptive Learning Rate + Bias Correction

Imagine driving a high-tech sports car equipped with AI sensors:
* **Momentum (1st Moment):** Acts like the car's engine **momentum**. If you have been driving downhill in the same direction, it builds up speed so you can cross small bumps and shallow valleys effortlessly.
* **RMSProp (2nd Moment):** Acts like an **adaptive suspension system**. If the road ahead gets extremely bumpy (large recent gradients), it stiffens the suspension (shrinks the step size) to prevent crashing. If the road is smooth (small gradients), it softens the suspension to speed up.
* **Bias Correction:** Acts like a **warm-up system**. When you start the engine (step 1), the memory sensors start at zero. Bias correction recalculates the initial sensor readings so the car doesn't jerk or stall during the first few seconds of driving.

## 2. The Mathematics: How Adam Works

At each training step $t$, Adam updates every weight $w$ using four sequential steps:

### Step 1: Update the 1st Moment (Exponential Moving Average of Gradients - Momentum)
$$m_t = \beta_1 \cdot m_{t-1} + (1 - \beta_1) \cdot g_t$$

### Step 2: Update the 2nd Moment (Exponential Moving Average of Squared Gradients - RMSProp)
$$v_t = \beta_2 \cdot v_{t-1} + (1 - \beta_2) \cdot g_t^2$$

### Step 3: Compute Bias-Corrected Moments
Because $m_0 = 0$ and $v_0 = 0$, $m_t$ and $v_t$ are biased towards zero at the start. We correct this by dividing by $(1 - \beta^t)$:

$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}$$

$$\hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$

### Step 4: Update the Weight
$$w_{t+1} = w_t - \frac{\eta}{\sqrt{\hat{v}_t} + \epsilon} \cdot \hat{m}_t$$

### Symbol Legend:
* **$w_t$:** Current weight value at step $t$.
* **$g_t$:** Raw gradient for this weight at the current step.
* **$m_t$:** 1st moment (moving average of gradients).
* **$v_t$:** 2nd moment (moving average of squared gradients).
* **$\hat{m}_t$ & $\hat{v}_t$:** Bias-corrected 1st and 2nd moments.
* **$\beta_1$ (Beta 1):** Decay rate for 1st moment (default **`0.9`**).
* **$\beta_2$ (Beta 2):** Decay rate for 2nd moment (default **`0.999`**).
* **$\beta_1^t$ & $\beta_2^t$:** $\beta_1$ and $\beta_2$ raised to the power of step number $t$.
* **$\eta$ (Eta):** Global base learning rate (default **`0.001`**).
* **$\epsilon$ (Epsilon):** A tiny constant (like $10^{-7}$ or $10^{-8}$) to prevent division by zero.

## 3. Comparison Across All Major Optimizers

| Feature | SGD | SGD + Momentum | AdaGrad | RMSProp | Adam |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Momentum (1st Moment)** | No | Yes | No | No | **Yes** |
| **Adaptive LR (2nd Moment)** | No | No | Yes (Cumulative Sum) | Yes (Moving Average) | **Yes (Moving Average)** |
| **Bias Correction** | No | No | No | No | **Yes** |
| **Memory Overhead** | None | 1 State ($v_t$) | 1 State ($S_t$) | 1 State ($v_t$) | **2 States ($m_t, v_t$)** |
| **Learning Rate Freezing** | No | No | High | None | **None** |
| **Default Benchmark** | Basic | Fast Descent | Sparse Data | RNNs / Vision | **Universal Default** |

## 4. Where is Adam Used?

Adam is the most widely adopted optimizer across all of machine learning and deep learning:
* **Transformers & LLMs (GPT, LLaMA, BERT):** Standard choice for training massive language models.
* **Computer Vision (CNNs, Vision Transformers):** Excellent for image classification, object detection, and segmentation.
* **General Deep Learning (MLPs, Autoencoders):** Default starting optimizer for almost any neural network architecture.

## 5. In Keras Code

```python
from tensorflow import keras

# Define Adam optimizer with standard default parameters
opt = keras.optimizers.Adam(
    learning_rate=0.001,
    beta_1=0.9,         # these are the universally used value for beta_1, but can also be changed
    beta_2=0.999,       # these are the universally used value for beta_2, but can also be changed
    epsilon=1e-7
)

# Compile model
model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])
```

---

# Cons (Disadvantages) of the Adam Optimizer

## 1. High Memory Overhead
* **The Problem:** Adam must maintain **two additional state vectors** ($m_t$ for 1st moment and $v_t$ for 2nd moment) for **every single weight** in the neural network.
* **Why it hurts:** 
  * Standard SGD requires $0$ extra optimizer memory.
  * Momentum / RMSProp require $1\times$ extra parameter memory.
  * **Adam requires $2\times$ extra parameter memory.**
* **Impact:** For large models (like a 7-billion parameter LLM using 32-bit floats), storing Adam's optimizer states alone requires **56 GB of GPU VRAM** before even loading the model weights or activations!


## 2. Inferior Generalization Compared to SGD with Momentum
* **The Problem:** Extensive empirical research shows that on certain tasks (especially Computer Vision tasks like training ResNets on ImageNet), **SGD with Momentum often achieves slightly better final test accuracy** than Adam.
* **Why it hurts:** Because Adam aggressively adapts learning rates per parameter, it can sometimes get trapped in sharp, narrow local minima that overfit to the training data, whereas SGD + Momentum naturally settles into broader, more generalizable flat minima.


## 3. Weight Decay Bug (Fixed by AdamW)
* **The Problem:** In standard Adam, adding traditional L2 regularization (weight decay) inside the loss function does **not** behave like true weight decay.
* **Why it hurts:** The L2 penalty gradient gets scaled down by $\frac{1}{\sqrt{\hat{v}_t + \epsilon}}$. As a result, weights with large recent gradients receive *less* weight decay than weights with small gradients.
* **Solution:** Researchers had to invent **AdamW** (which decouples weight decay from the gradient update step) to fix this flaw.


## 4. Potential Instability with Extreme Outlier Gradients
* **The Problem:** In non-stationary environments or noisy mini-batches with extreme outliers, the 2nd moment $v_t$ can rapidly shrink or fluctuate.
* **Why it hurts:** If $v_t$ drops suddenly while $m_t$ remains large, the step size $\frac{\eta}{\sqrt{\hat{v}_t} + \epsilon} \cdot \hat{m}_t$ can spike unpredictably, causing gradient explosion or instability.
* *(Note: Variants like AMSGrad and RAdam were introduced to address this specific issue).*


## 5. More Hyperparameters to Manage
* While Adam's defaults ($\eta = 0.001$, $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\epsilon = 10^{-7}$) work well out-of-the-box for most tasks, if the defaults fail on a custom architecture, you now have **4 different hyperparameters** to tune simultaneously:
  1. Learning rate ($\eta$)
  2. 1st moment decay ($\beta_1$)
  3. 2nd moment decay ($\beta_2$)
  4. Numerical stability constant ($\epsilon$)

---

# The Mathematics of the Adam Optimizer

Adam (Adaptive Moment Estimation) tracks both the first moment (the mean of gradients) and the second moment (the uncentered variance of gradients). It includes explicit bias correction terms to compensate for zero initialization in early steps.

## 1. The Core Equations

At each training step $t$, the optimizer updates every weight $w$ using four mathematical steps:

### Step 1: Update 1st Moment (Exponential Moving Average of Gradients)

$$m_t = \beta_1 \cdot m_{t-1} + (1 - \beta_1) \cdot g_t$$

### Step 2: Update 2nd Moment (Exponential Moving Average of Squared Gradients)

$$v_t = \beta_2 \cdot v_{t-1} + (1 - \beta_2) \cdot g_t^2$$

### Step 3: Compute Bias-Corrected Moments

$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}$$

$$\hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$

### Step 4: Update the Weight

$$w_{t+1} = w_t - \frac{\eta}{\sqrt{\hat{v}_t} + \epsilon} \cdot \hat{m}_t$$

## 2. Symbol Legend

* **$w_t$:** The weight value at the current step $t$.
* **$w_{t+1}$:** The updated weight value for the next step.
* **$g_t$:** The gradient calculated for the weight at step $t$.
* **$m_t$:** The 1st moment vector (moving average of gradients).
* **$v_t$:** The 2nd moment vector (moving average of squared gradients).
* **$\hat{m}_t$:** The bias-corrected 1st moment vector.
* **$\hat{v}_t$:** The bias-corrected 2nd moment vector.
* **$\beta_1$ (Beta 1):** Decay factor for 1st moment (default $0.9$).
* **$\beta_2$ (Beta 2):** Decay factor for 2nd moment (default $0.999$).
* **$\beta_1^t$ & $\beta_2^t$:** Decay factors raised to the power of step $t$.
* **$\eta$ (Eta):** Global base learning rate (default $0.001$).
* **$\epsilon$ (Epsilon):** A tiny constant (e.g., $10^{-7}$) added to prevent division by zero.

## 3. Step-by-Step Mathematical Walkthrough

Let me calculate the updates for a single weight over 3 steps using:
* **Starting Weight ($w_1$):** $0.5$
* **Starting 1st Moment ($m_0$):** $0.0$
* **Starting 2nd Moment ($v_0$):** $0.0$
* **Decay Rates:** $\beta_1 = 0.9$, $\beta_2 = 0.999$
* **Learning Rate ($\eta$):** $0.1$
* **Epsilon ($\epsilon$):** Negligible for this calculation

### Step 1 ($t = 1$, Current Gradient $g_1 = 2.0$)

**1. Calculate raw moments:**
$$m_1 = 0.9(0.0) + 0.1(2.0) = \mathbf{0.2}$$
$$v_1 = 0.999(0.0) + 0.001(2.0)^2 = 0.001 \cdot 4.0 = \mathbf{0.004}$$

**2. Apply bias correction ($t=1$):**
$$\hat{m}_1 = \frac{0.2}{1 - (0.9)^1} = \frac{0.2}{0.1} = \mathbf{2.0}$$
$$\hat{v}_1 = \frac{0.004}{1 - (0.999)^1} = \frac{0.004}{0.001} = \mathbf{4.0}$$

**3. Update weight ($w_2$):**
$$w_2 = w_1 - \frac{\eta}{\sqrt{\hat{v}_1}} \cdot \hat{m}_1 = 0.5 - \left( \frac{0.1}{\sqrt{4.0}} \cdot 2.0 \right) = 0.5 - (0.05 \cdot 2.0) = \mathbf{0.4}$$

---

### Step 2 ($t = 2$, Current Gradient $g_2 = 1.0$)

**1. Calculate raw moments:**
$$m_2 = 0.9(0.2) + 0.1(1.0) = 0.18 + 0.10 = \mathbf{0.28}$$
$$v_2 = 0.999(0.004) + 0.001(1.0)^2 = 0.003996 + 0.001000 = \mathbf{0.004996}$$

**2. Apply bias correction ($t=2$):**
$$\beta_1^2 = (0.9)^2 = 0.81 \implies \hat{m}_2 = \frac{0.28}{1 - 0.81} = \frac{0.28}{0.19} \approx \mathbf{1.4737}$$
$$\beta_2^2 = (0.999)^2 = 0.998001 \implies \hat{v}_2 = frac{0.004996}{1 - 0.998001} = \frac{0.004996}{0.001999} \approx \mathbf{2.4992}$$

**3. Update weight ($w_3$):**
$$\sqrt{\hat{v}_2} = \sqrt{2.4992} \approx 1.5809$$
$$w_3 = w_2 - \left( \frac{0.1}{1.5809} \cdot 1.4737 \right) \approx 0.4 - (0.06325 \cdot 1.4737) = 0.4 - 0.0932 = \mathbf{0.3068}$$

---

### Step 3 ($t = 3$, Current Gradient $g_3 = 0.5$)

**1. Calculate raw moments:**
$$m_3 = 0.9(0.28) + 0.1(0.5) = 0.252 + 0.050 = \mathbf{0.302}$$
$$v_3 = 0.999(0.004996) + 0.001(0.5)^2 = 0.004991 + 0.000250 = \mathbf{0.005241}$$

**2. Apply bias correction ($t=3$):**
$$\beta_1^3 = (0.9)^3 = 0.729 \implies \hat{m}_3 = \frac{0.302}{1 - 0.729} = \frac{0.302}{0.271} \approx \mathbf{1.1144}$$
$$\beta_2^3 = (0.999)^3 = 0.997003 \implies \hat{v}_3 = \frac{0.005241}{1 - 0.997003} = \frac{0.005241}{0.002997} \approx \mathbf{1.7487}$$

**3. Update weight ($w_4$):**
$$\sqrt{\hat{v}_3} = \sqrt{1.7487} \approx 1.3224$$
$$w_4 = w_3 - \left( \frac{0.1}{1.3224} \cdot 1.1144 \right) \approx 0.3068 - (0.07562 \cdot 1.1144) = 0.3068 - 0.0843 = \mathbf{0.2225}$$

## 4. Why Bias Correction is Crucial in Early Steps

Look at what would happen at Step 1 without bias correction:
* Raw 1st moment: $m_1 = 0.2$
* Raw 2nd moment: $v_1 = 0.004$
* Without bias correction, $v_1 = 0.004$ is artificially tiny because $v_0 = 0$. $\sqrt{v_1} = 0.0632$. This would cause a massive, uncalibrated initial jump.

By applying bias correction $\hat{m}_t = \frac{m_t}{1-\beta_1^t}$ and $\hat{v}_t = \frac{v_t}{1-\beta_2^t}$:
* At step 1 ($t=1$), $1 - beta_1^1 = 0.1$ and $1 - beta_2^1 = 0.001$.
* Dividing by these small values scales up $m_1$ to $2.0$ and $v_1$ to $4.0$.

As step $t$ grows large ($t \to \infty$), $\beta_1^t \to 0$ and $\beta_2^t \to 0$, so $(1 - \beta^t) \to 1$. The bias correction factors naturally fade away, leaving pure uncorrected moments once the moving averages have warmed up!